In [2]:
import pandas as pd
import os
from tqdm.auto import tqdm

In [3]:
def get_text(path: str) -> str:

    with open(path, 'r' ,encoding='utf-8') as f: 
        text = f.read()

    return text

df = pd.read_json('norec\data\metadata.json', encoding='utf-8')
data = df.T[['id', 'rating']]
data['txt_names'] = ['0'*(6-len(str(id))) + str(id) + '.txt' for id in data['id']]
ids = data['txt_names']
paths = []
texts = []
splits = []

for root, dirs, files in os.walk("norec\\data", topdown=False):
   for name in files:
       path = os.path.join(root, name)
       if path.endswith('.txt'):
           paths.append(path)

for id in tqdm(ids):
    for path in paths:
        if id == path.split('\\')[-1]:
            texts.append(get_text(path))
            splits.append(path.split('\\')[-2])


data['text'] = texts
data['split'] = splits
data = data.rename(columns={"rating": "label"}, errors="raise")
data['label'] = [x-1 for x in data['label']]
data.to_csv('combined_data.csv', index=False)

100%|██████████| 43614/43614 [22:10<00:00, 32.77it/s]


In [1]:
import transformers
from transformers import BertModel, BertTokenizer, AdamW, LongformerModel, get_linear_schedule_with_warmup, AutoTokenizer, LongformerTokenizer
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from collections import defaultdict
from textwrap import wrap
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import torch.nn.functional as F
import warnings
warnings.filterwarnings("ignore")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


#PRE_TRAINED_MODEL_NAME = 'ltgoslo/norbert2'
PRE_TRAINED_MODEL_NAME = 'allenai/longformer-base-4096'

MAX_LEN = 2048   # 4096   # longest text = 9111
RANDOM_SEED = 42
BATCH_SIZE = 2
EPOCHS = 10

df = pd.read_csv('combined_data.csv')
class_names = df.label.unique()
tokenizer = LongformerTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

class Dataset(Dataset):
  def __init__(self, texts, targets, tokenizer, max_len):
    self.texts = texts
    self.targets = targets
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, item):
    text = str(self.texts[item])
    target = self.targets[item]
    encoding = self.tokenizer.encode_plus(
      text,
      add_special_tokens=True,
      max_length=self.max_len,
      return_token_type_ids=False,
      pad_to_max_length=True,
      return_attention_mask=True,
      return_tensors='pt',
      truncation=True
    )
    return {
      'text': text,
      'input_ids': encoding['input_ids'].flatten(),
      'attention_mask': encoding['attention_mask'].flatten(),
      'targets': torch.tensor(target, dtype=torch.long)
    }


def create_data_loader(df, tokenizer, max_len, batch_size):
  ds = Dataset(
    texts=df.text.to_numpy(),
    targets=df.label.to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
  )
  return DataLoader(
    ds,
    batch_size=batch_size
  )

class SentimentClassifier(nn.Module):

  def __init__(self, n_classes):
    super(SentimentClassifier, self).__init__()
    self.long_f = LongformerModel.from_pretrained(PRE_TRAINED_MODEL_NAME)
    self.drop = nn.Dropout(p=0.3)
    self.out = nn.Linear(self.long_f.config.hidden_size, n_classes)

  def forward(self, input_ids, attention_mask):
    
    long_f_output = self.long_f(
      input_ids=input_ids,
      attention_mask=attention_mask,
      return_dict=False
    )
    last_hidden_state, pooled_output = long_f_output
    output = self.drop(pooled_output)
    return self.out(output)

df_train, df_val = train_test_split(
  df,
  test_size=0.1,
  random_state=RANDOM_SEED
)
train_data_loader = create_data_loader(df_train, tokenizer, MAX_LEN, BATCH_SIZE)
val_data_loader = create_data_loader(df_val, tokenizer, MAX_LEN, BATCH_SIZE)


In [7]:

for x in tqdm(train_data_loader):
  
  x['attention_mask'] = torch.tensor(x['attention_mask'], dtype=torch.int16)
  x['input_ids'] = torch.tensor(x['input_ids'], dtype=torch.int16)


 11%|█         | 2090/19626 [00:47<06:38, 43.99it/s]


KeyboardInterrupt: 